# Reading the Trail: Colab runner

Runs the full experiment matrix (E0–E9) with `scripts/run_all.py`. Checkpoints, logs and decisions are saved to Google Drive, so if the session disconnects, **just run all cells again**: finished runs are skipped and an interrupted run resumes from its last epoch.

1. Runtime → Change runtime type → **GPU** (T4 on free Colab; L4/A100 on Colab Pro).
2. Runtime → Run all.
3. Keep this tab open. Free Colab ends sessions after ~12 h or when idle.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os; print('CPUs:', os.cpu_count())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/reading-the-trail'   # runs/ and results/ are saved here
REPO_URL = 'https://github.com/cnash03/Deep-Learning-Project.git'
BRANCH = 'experiments'
!mkdir -p {PROJECT}/runs {PROJECT}/results

In [ ]:
%cd /content
!test -d repo || git clone -q -b $BRANCH $REPO_URL repo
%cd /content/repo
!git fetch -q origin $BRANCH && git checkout -q $BRANCH && git reset -q --hard origin/$BRANCH && git log --oneline -1
# Keep Colab's own torch/numpy/etc.; install only what Colab lacks (versions match requirements.txt).
!pip install -q segmentation_models_pytorch==0.6.0 timm==1.0.31 albumentations==2.0.8

## Data
RUGD (~5.7 GB) is downloaded to the Colab machine's local disk each session (a few minutes). It is not stored on Drive, so it doesn't use your Drive quota.

In [ ]:
import os
os.environ['DATA_ROOT'] = '/content/data/rugd'
if not os.path.exists('/content/data/rugd/RUGD_annotations'):
    !bash scripts/download_rugd.sh /content/data/rugd && rm -f /content/data/rugd/*.zip
!python -m scripts.check_splits --root /content/data/rugd

## Train everything
This cell runs for many hours. Progress per epoch is in `runs/<name>.log` on Drive; the next cell shows a summary.

In [ ]:
WORKERS = min(8, os.cpu_count())
!python -W ignore -m scripts.run_all --runs-dir {PROJECT}/runs --results-dir {PROJECT}/results --set train.num_workers={WORKERS}

## Progress (safe to run any time, even from a second notebook)

In [ ]:
!tail -n 15 {PROJECT}/runs/orchestrator.log
!for f in {PROJECT}/runs/*.log; do echo "$(basename $f): $(grep -E '^\[' $f | tail -1)"; done